In [171]:
import tensorflow as tf
import tensorflow_decision_forests as tfdf
from tensorflow.keras.layers import Dense, Dropout, Layer, Input, LayerNormalization
from tensorflow.keras.models import Model, load_model
import numpy as np
from transformers import TFBertModel, BertTokenizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
import logging
import matplotlib.pyplot as plt
import pandas as pd
from lime.lime_tabular import LimeTabularExplainer
from lime.lime_text import LimeTextExplainer
import shap
from collections import defaultdict
import re

In [172]:
# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')


In [173]:
cali_housing_path = '../data/California_Houses.csv'
RANDOM_SEED = 492
cali_df = pd.read_csv(cali_housing_path)
y_series = cali_df['Median_House_Value']
y = pd.DataFrame(y_series, columns=['Median_House_Value'])
features = [col for col in cali_df.columns if col != 'Median_House_Value']
X = cali_df[features]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED)

X_train = X_train.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)


In [174]:
class FeatureTokenizer(tf.keras.layers.Layer):
    def __init__(self, num_features, embed_dim):
        super(FeatureTokenizer, self).__init__()
        self.num_features = num_features
        self.embed_dim = embed_dim
        
        # Element-wise multiplication weights for each feature
        self.feature_weights = self.add_weight(shape=(num_features, embed_dim),
                                               initializer='random_normal',
                                               trainable=True)
        # Bias for each feature
        self.feature_bias = self.add_weight(shape=(num_features, embed_dim),
                                            initializer='random_normal',
                                            trainable=True)
        # Layer normalization
        self.layer_norm = LayerNormalization()
    
    def call(self, inputs):
        embeddings = inputs[..., tf.newaxis] * self.feature_weights + self.feature_bias
        embeddings = self.layer_norm(embeddings)
        return embeddings

# Example usage:
num_features = 13  # Number of numerical features
embed_dim = 16      # Embedding dimension

feature_tokenizer = FeatureTokenizer(num_features, embed_dim)

In [189]:
# make the data into strings
X_train_texts = X_train_scaled.astype(str).apply(' '.join, axis=1).tolist()
X_test_texts = X_test_scaled.astype(str).apply(' '.join, axis=1).tolist()

In [190]:
# Load the pre-trained BERT model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = TFBertModel.from_pretrained('bert-base-uncased')

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.predictions.bias', 'cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.LayerNorm.bias']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertModel for predictions w

In [191]:
# Function to concatenate column names and values
def concatenate_columns_with_names(df):
    concatenated = df.apply(lambda row: ' '.join([f"{col} {val}" for col, val in row.items()]), axis=1)
    return concatenated.tolist()
# Concatenate column names and values for each instance
X_train_texts_with_names = concatenate_columns_with_names(X_train_scaled)
X_test_texts_with_names = concatenate_columns_with_names(X_test_scaled)

In [192]:
# Tokenize and encode inputs
max_length = 230  # Define maximum sequence length
train_encodings = tokenizer(X_train_texts_with_names, truncation=True, padding=True, max_length=max_length, return_tensors='tf')
test_encodings = tokenizer(X_test_texts_with_names, truncation=True, padding=True, max_length=max_length, return_tensors='tf')

In [193]:
# Convert train_encodings and test_encodings to numpy arrays
X_train_input_ids = train_encodings['input_ids'].numpy()
X_train_attention_mask = train_encodings['attention_mask'].numpy()
X_train_token_type_ids = train_encodings['token_type_ids'].numpy()

X_test_input_ids = test_encodings['input_ids'].numpy()
X_test_attention_mask = test_encodings['attention_mask'].numpy()
X_test_token_type_ids = test_encodings['token_type_ids'].numpy()

In [175]:
class MLP(tf.keras.Model):
    def __init__(self, max_length, input_dim, hidden_dim, dropout_rate=0.1):
        super(MLP, self).__init__()
        self.max_length = max_length
        self.bert_model = TFBertModel.from_pretrained('bert-base-uncased')
        self.bert_model.trainable = True
        self.dropout1 = Dropout(dropout_rate)
        self.dense1 = Dense(hidden_dim, activation='relu')
        self.dropout2 = Dropout(dropout_rate)
        self.dense2 = Dense(hidden_dim, activation='relu')
        self.dropout3 = Dropout(dropout_rate)
        self.dense3 = Dense(1, activation='linear')

    def call(self, inputs, training=False):
        input_ids, attention_mask, token_type_ids = inputs
        bert_output = self.bert_model(input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)[1]
        # Apply dropout
        x = self.dropout1(bert_output, training=training)
        x = self.dense1(x)
        x = self.dropout2(x, training=training)
        x = self.dense2(x)
        x = self.dropout3(x, training=training)
        x = self.dense3(x)
        return x

In [176]:
class DropConnectDense(tf.keras.layers.Layer):
    def __init__(self, units, dropout_rate=0.1, **kwargs):
        super(DropConnectDense, self).__init__(**kwargs)
        self.units = units
        self.dropout_rate = dropout_rate
        self.dense = Dense(units)

    def build(self, input_shape):
        self.kernel = self.add_weight("kernel", shape=[input_shape[-1], self.units])
        self.bias = self.add_weight("bias", shape=[self.units])

    def call(self, inputs, training=False):
        if training:
            weights = self.dense.kernel * tf.keras.backend.random_binomial(
                shape=self.dense.kernel.shape, p=1 - self.dropout_rate)
            output = tf.keras.backend.dot(inputs, weights)
            if self.dense.use_bias:
                output = tf.keras.backend.bias_add(output, self.dense.bias)
            return self.dense.activation(output)
        else:
            return self.dense(inputs)

class MLPDropConnect(tf.keras.Model):
    def __init__(self, max_length, input_dim, hidden_dim, dropout_rate=0.1):
        super(MLPDropConnect, self).__init__()
        self.max_length = max_length
        self.bert_model = TFBertModel.from_pretrained('bert-base-uncased')
        self.bert_model.trainable = True
        self.dense1 = DropConnectDense(hidden_dim, dropout_rate)
        self.dense2 = DropConnectDense(hidden_dim, dropout_rate)
        self.dense3 = Dense(1, activation='linear')
    
    def call(self, inputs, training=False):
        input_ids, attention_mask, token_type_ids = inputs
        bert_output = self.bert_model(input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)[1]
        x = self.dense1(bert_output, training=training)
        x = self.dense2(x, training=training)
        x = self.dense3(x)
        return x

In [ ]:
    # text_instance = tokenizer.decode(instance, skip_special_tokens=True)
    # explainer = LimeTextExplainer(class_names=['predicted_value'])
    # explanation = explainer.explain_instance(text_instance, model.predict, num_features=num_features)

In [134]:
# def explain_lime(model, instance, train_data, num_features):
#     explainer = LimeTabularExplainer(
#         training_data=np.array(train_data),
#         feature_names=[f'feature_{i}' for i in range(train_data.shape[1])],
#         mode='regression'
#     )
#     explanation = explainer.explain_instance(instance, model.predict, num_features=num_features)
#     return explanation

In [177]:
def explain_lime(model, instance, train_data, num_features):
    explainer = LimeTabularExplainer(
        training_data=np.array(train_data),
        feature_names=[f'feature_{i}' for i in range(train_data.shape[1])],
        mode='regression'
    )
    tabular_instance = instance.numpy().reshape(1, -1)  # Convert to tabular format
    explanation = explainer.explain_instance(tabular_instance[0], model.predict, num_features=num_features)
    return explanation

In [135]:
# def explain_shap(model, instance, train_data):
#     explainer = shap.KernelExplainer(model.predict, train_data)
#     shap_values = explainer.shap_values(instance.numpy())
#     return shap_values

In [178]:
def explain_shap(model, instance, train_data):
    explainer = shap.KernelExplainer(model.predict, train_data)
    tabular_instance = instance.numpy().reshape(1, -1)  # Convert to tabular format
    shap_values = explainer.shap_values(tabular_instance)
    return shap_values


In [0]:
# def predict_with_explanation(model, inputs, train_data, num_samples=10, num_features=13):
#     # Add batch dimension for each input tensor
#     input_ids = tf.expand_dims(inputs[0], axis=0)
#     attention_mask = tf.expand_dims(inputs[1], axis=0)
#     token_type_ids = tf.expand_dims(inputs[2], axis=0)
#     
#     predictions = []
#     lime_explanations = []
#     shap_explanations = []
#     for _ in range(num_samples):
#         prediction = model([input_ids, attention_mask, token_type_ids], training=True)
#         predictions.append(prediction)
#         
#         lime_explanation = explain_lime(model, input_ids, train_data, num_features)
#         shap_explanation = explain_shap(model, input_ids, train_data)
#         lime_explanations.append(lime_explanation)
#         shap_explanations.append(shap_explanation)
#     predictions = tf.stack(predictions, axis=0)
#     prediction_mean = tf.reduce_mean(predictions, axis=0)
#     prediction_std = tf.math.reduce_std(predictions, axis=0)
#     return prediction_mean.numpy(), prediction_std.numpy(), lime_explanations, shap_explanations

In [179]:
def predict_with_explanation(model, inputs, num_samples=10, num_features=13):
    input_ids = tf.expand_dims(inputs[0], axis=0)
    attention_mask = tf.expand_dims(inputs[1], axis=0)
    token_type_ids = tf.expand_dims(inputs[2], axis=0)
    
    predictions = []
    lime_explanations = []
    shap_explanations = []
    for _ in range(num_samples):
        prediction = model([input_ids, attention_mask, token_type_ids], training=True)
        predictions.append(prediction)
        
        lime_explanation = explain_lime(model, input_ids, prediction, num_features)
        shap_explanation = explain_shap(model, input_ids, prediction)
        
        lime_explanations.append(lime_explanation)
        shap_explanations.append(shap_explanation)
    predictions = tf.stack(predictions, axis=0)
    prediction_mean = tf.reduce_mean(predictions, axis=0)
    prediction_std = tf.math.reduce_std(predictions, axis=0)
    return prediction_mean.numpy(), prediction_std.numpy(), lime_explanations, shap_explanations


In [180]:

def predict_with_uncertainty(model, inputs, num_samples=10):
    # Add batch dimension for each input tensor
    input_ids = tf.expand_dims(inputs[0], axis=0)
    attention_mask = tf.expand_dims(inputs[1], axis=0)
    token_type_ids = tf.expand_dims(inputs[2], axis=0)
    
    predictions = []
    for _ in range(num_samples):
        prediction = model([input_ids, attention_mask, token_type_ids], training=True)
        predictions.append(prediction)
    predictions = tf.stack(predictions, axis=0)
    prediction_mean = tf.reduce_mean(predictions, axis=0)
    prediction_std = tf.math.reduce_std(predictions, axis=0)
    return prediction_mean.numpy(), prediction_std.numpy()

In [181]:
class ModelWithUncertainty(Model):
    def __init__(self, input_dim, hidden_dim, dropout_rate=0.1, method='mc_dropout'):
        super(ModelWithUncertainty, self).__init__()
        self.method = method
        if method == 'mc_dropout':
            self.model = MLP(max_length=input_dim, input_dim=input_dim, hidden_dim=hidden_dim, dropout_rate=dropout_rate)
        elif method == 'dropconnect':
            self.model = MLPDropConnect(max_length=input_dim, input_dim=input_dim, hidden_dim=hidden_dim, dropout_rate=dropout_rate)
        else:
            raise ValueError("Method should be 'mc_dropout' or 'dropconnect'")
    
    def call(self, inputs, training=False):
        return self.model(inputs, training=training)
    
    def predict_with_explanation(self, inputs, num_samples):
        return predict_with_explanation(self.model, inputs, num_samples)
    
    def predict_with_uncertainty(self, inputs, num_samples):
        return predict_with_uncertainty(self.model, inputs, num_samples)

    def explain_lime(self, instance, data, num_features):
        return explain_lime(self.model, instance, data, num_features=num_features)

    def explain_shap(self, instance, data):
        return explain_shap(self.model, instance, data)

In [182]:
dense_size = 300
dropout_rate = 0.1
num_samples = 20
num_features = 13
explain_instance = 3

In [194]:
model = ModelWithUncertainty(input_dim=num_features, hidden_dim=dense_size, method='mc_dropout')
model.compile(optimizer='adam', loss='mse')

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.predictions.bias', 'cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.LayerNorm.bias']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertModel for predictions w

In [195]:
history_unfreeze = model.fit(
    train_encodings, 
    y_train, 
    epochs=1, 
    batch_size=32, 
    validation_split=0.2
)

ValueError: Argument `validation_split` is only supported for tensors or NumPy arrays.Found incompatible type in the input: [<class 'transformers.tokenization_utils_base.BatchEncoding'>]

In [48]:
# Freeze the BERT model
model.model.bert_model.trainable = False

In [50]:
# Print the trainable status of each layer in the BERT model
for layer in model.model.bert_model.layers:
    print(f'Layer: {layer.name}, Trainable: {layer.trainable}')

Layer: bert, Trainable: False


In [49]:
history_frozen = model.fit(
    [X_train_input_ids, X_train_attention_mask, X_train_token_type_ids], 
    y_train, 
    epochs=30,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/30
 26/413 ━━━━━━━━━━━━━━━━━━━━ 37:27 6s/step - loss: 14228024320.0000

KeyboardInterrupt: 

In [ ]:
# Concatenate the histories
history = {}
for key in history_unfreeze.history.keys():
    history[key] = history_unfreeze.history[key] + history_frozen.history[key]

# Plot the training and validation loss
plt.figure(figsize=(12, 6))
plt.plot(history['loss'], label='Training Loss')
plt.plot(history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [22]:
print(train_encodings)

{'input_ids': <tf.Tensor: shape=(16512, 229), dtype=int32, numpy=
array([[ 101, 3991, 1035, ...,    0,    0,    0],
       [ 101, 3991, 1035, ...,    0,    0,    0],
       [ 101, 3991, 1035, ...,    0,    0,    0],
       ...,
       [ 101, 3991, 1035, ...,    0,    0,    0],
       [ 101, 3991, 1035, ...,    0,    0,    0],
       [ 101, 3991, 1035, ...,    0,    0,    0]], dtype=int32)>, 'token_type_ids': <tf.Tensor: shape=(16512, 229), dtype=int32, numpy=
array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], dtype=int32)>, 'attention_mask': <tf.Tensor: shape=(16512, 229), dtype=int32, numpy=
array([[1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       ...,
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0]], dtype=int32)>}


In [151]:
combined_indices = [1112, 2904, 4006, 2905, 1326, 128]
X_test_input_ids_selected = []
X_test_attention_mask_selected = []
X_test_token_type_ids_selected = []
for i in combined_indices:
    X_test_input_ids_selected.append(X_test_input_ids[i])
    X_test_attention_mask_selected.append(X_test_attention_mask[i])
    X_test_token_type_ids_selected.append(X_test_token_type_ids[i])
X_test_input_ids_selected = np.array(X_test_input_ids_selected)
X_test_attention_mask_selected = np.array(X_test_attention_mask_selected)
X_test_token_type_ids_selected = np.array(X_test_token_type_ids_selected)


In [162]:
train_data = X_train_scaled
# Initialize lists to store results
predicted_means = []
predicted_uncertainties = []
lime_explanations = []
shap_explanations = []
num_samples = 10  # Example number of samples for uncertainty estimation
num_features = X_train_scaled.shape[1]  # Number of features


In [168]:
for i in range(len(combined_indices)):
    # Prepare inputs for the instance
    inputs = [
        X_test_input_ids_selected[i],
        X_test_attention_mask_selected[i],
        X_test_token_type_ids_selected[i]
    ]
    
    # Get prediction with explanation
    prediction_mean, prediction_std, lime_explanation, shap_explanation = model.predict_with_explanation(
        inputs, train_data
    )
    
    # Append results to the lists
    predicted_means.append(prediction_mean)
    predicted_uncertainties.append(prediction_std)
    lime_explanations.append(lime_explanation)
    shap_explanations.append(shap_explanation)

# Convert results to arrays or any preferred format
predicted_means = np.array(predicted_means)
predicted_uncertainties = np.array(predicted_uncertainties)

TypeError: list indices must be integers or slices, not str

In [140]:
bert_cali_predicted_mean_instance, bert_cali_predicted_uncertainty_instance = model.predict_with_uncertainty([X_test_input_ids_selected[1], X_test_attention_mask_selected[1], X_test_token_type_ids_selected[1]], num_samples)


In [141]:
print(bert_cali_predicted_mean_instance)

[[214149.34]]


In [152]:

inputs = {
    'input_ids': X_test_input_ids_selected[0],
    'attention_mask': X_test_attention_mask_selected[0],
    'index': 0  # Index to locate the corresponding tabular data instance
}

bert_cali_predicted_mean_instance, bert_cali_predicted_uncertainty_instance, bert_cali_explain_lime_instance, bert_cali_selected_explain_shap_instance = model.predict_with_explanation(
    inputs, 
    num_samples
)

KeyError: 0

In [112]:
print(bert_cali_explain_lime_instance)

NameError: name 'bert_cali_explain_lime_instance' is not defined

In [93]:
bert_cali_selected_predicted = []
bert_cali_selected_predicted_uncertainty = []
bert_cali_selected_explain_lime = []
bert_cali_selected_explain_shap = []
for i in range(len(combined_indices)):  
    bert_cali_predicted_mean_instance, bert_cali_predicted_uncertainty_instance, bert_cali_explain_lime_instance, bert_cali_selected_explain_shap_instance = model.predict_with_explanation([X_test_input_ids_selected[i], X_test_attention_mask_selected[i], X_test_token_type_ids_selected[i]], num_samples)
    bert_cali_selected_predicted.append(bert_cali_predicted_mean_instance)
    bert_cali_selected_predicted_uncertainty.append(bert_cali_predicted_uncertainty_instance)
    bert_cali_selected_explain_lime.append(bert_cali_explain_lime_instance)
    bert_cali_selected_explain_shap.append(bert_cali_selected_explain_shap_instance)

ValueError: Unrecognized data type: x=['median _ income 0. 20439311053278844 median _ age - 0. 29425674313362365 tot _ rooms 1. 5326130911890214 tot _ bedrooms 1. 470085762327261 population 2. 737770794883824 households 1. 4856362805571295 latitude - 0. 8637994016415769 longitude 0. 8815007206375519 distance _ to _ coast - 0. 35371763569232817 distance _ to _ la - 0. 8879620686198497 distance _ to _ sandiego - 0. 9142280176883859 distance _ to _ sanjose 0. 8815189000235334 distance _ to _ sanfrancisco 0. 8874968522134921', 'median  income .  median   - . 29425674313362365    .    bedrooms .  population .   .   - .   .       - .       - .       - .       .       . ', 'median _  . 20439311053278844 median _  - .  tot _ rooms 1. 5326130911890214 tot _  1. 470085762327261  . 737770794883824  1.  latitude - . 8637994016415769 longitude .  distance _ to _  - .  distance _ to _ la - .  distance _ to _ sandiego - . 9142280176883859 distance _ to _ sanjose . 8815189000235334 distance _ to _ san

In [97]:
print(bert_cali_predicted_mean_instance)

NameError: name 'bert_cali_predicted_mean_instance' is not defined